# Notebook 3: Model Training

Loads the train/validation/test sets produced by `notebook_02_preprocessing` and trains models against them. This notebook does not repeat any cleaning or splitting logic, it only reads the already-processed CSVs, so it has no dependency on notebook 2's kernel state.

In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

## 1. Load Preprocessed Data

Reads the three partitions notebook 2 saved to disk. `SaleYearMonth` comes back as a plain string after the CSV round-trip (not a `Period`) — harmless here since it's excluded from the model's features either way, kept only for traceability back to which month each row belongs to.

In [7]:
os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

housing_train_m1 = pd.read_csv("CRMLSCleaned/housingtrainm1.csv")
housing_val_m1 = pd.read_csv("CRMLSCleaned/housingvalm1.csv")
housing_test_m1 = pd.read_csv("CRMLSCleaned/housingtestm1.csv")

print(f"train: {housing_train_m1.shape}")
print(f"val:   {housing_val_m1.shape}")
print(f"test:  {housing_test_m1.shape}")

train: (127053, 25)
val:   (12018, 25)
test:  (12015, 25)


## 2. Task 1: Linear Regression Baseline

A defensible baseline needs leakage-safe imputation and encoding, which is what `ColumnTransformer`/`Pipeline` are for — so this doubles as a first pass at Task 0's pipeline infrastructure, kept intentionally minimal since the goal here is a benchmark, not a tuned model.

`SaleYearMonth` is excluded from the feature list; it was only ever needed to make the chronological split possible, never as something the model should see as an input.

Categorical columns (`City`, `MLSAreaMajor`, `HighSchoolDistrict` in particular) are high-cardinality and go through plain one-hot encoding here as a known baseline simplification; CV-safe target encoding per the feature-engineering guidance is left as a future improvement.

R² on the test set is the metric the spec asks for; MAE, MAPE, and MdAPE are included alongside it since a single metric doesn't tell the whole story for skewed, multi-scale price data, and having them recorded now makes the eventual side-by-side comparison against advanced models possible without rerunning this cell later.

In [3]:
# SaleYearMonth: metadata carried through for the split, never a model input.
# ClosePrice: the target.
non_feature_columns = ["ClosePrice", "SaleYearMonth"]

numeric_feature_columns = [
    "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "GarageSpaces",
    "LivingArea", "LotSizeSquareFeet", "AssociationFee", "YearBuilt",
    "Levels", "Stories",
]
categorical_feature_columns = ["City", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict", "Flooring"]
feature_columns = numeric_feature_columns + categorical_feature_columns

# catches a typo or a column that's neither a listed feature nor listed metadata
assert set(feature_columns) == set(housing_train_m1.columns) - set(non_feature_columns), (
    "feature_columns doesn't match housing_train_m1's actual columns -- check for "
    "a typo, or a column that needs to be added to one list or the other."
)

preprocessor = ColumnTransformer(transformers=[
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_feature_columns),
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_feature_columns),
])

baseline_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression()),
])

X_train, y_train = housing_train_m1[feature_columns], housing_train_m1["ClosePrice"]
X_val, y_val = housing_val_m1[feature_columns], housing_val_m1["ClosePrice"]
X_test, y_test = housing_test_m1[feature_columns], housing_test_m1["ClosePrice"]

# fit on train only -- val/test are transformed using statistics learned from
# train alone, since preprocess is inside the same Pipeline that gets fit here
baseline_pipeline.fit(X_train, y_train)

def evaluate(pipeline, X, y, label):
    preds = pipeline.predict(X)
    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mape = mean_absolute_percentage_error(y, preds)
    mdape = np.median(np.abs((y - preds) / y))
    print(f"{label:>5s}: R2={r2:.4f}  MAE=${mae:,.0f}  MAPE={mape:.2%}  MdAPE={mdape:.2%}")
    return {"r2": r2, "mae": mae, "mape": mape, "mdape": mdape}

print("Baseline Linear Regression results:")
train_metrics = evaluate(baseline_pipeline, X_train, y_train, "train")
val_metrics = evaluate(baseline_pipeline, X_val, y_val, "val")
test_metrics = evaluate(baseline_pipeline, X_test, y_test, "test")

Baseline Linear Regression results:
train: R2=0.8397  MAE=$211,810  MAPE=20.45%  MdAPE=14.82%
  val: R2=0.0083  MAE=$480,550  MAPE=35.27%  MdAPE=15.43%
 test: R2=0.4413  MAE=$287,478  MAPE=25.26%  MdAPE=15.43%


In [5]:
baseline_results = {
    "model": "LinearRegression",
    "train": train_metrics,
    "val": val_metrics,
    "test": test_metrics,
}
baseline_results

{'model': 'LinearRegression',
 'train': {'r2': 0.8397218270724092,
  'mae': 211810.16393349884,
  'mape': 0.2044794781702773,
  'mdape': np.float64(0.14819261633428013)},
 'val': {'r2': 0.008344712174975455,
  'mae': 480550.48043509683,
  'mape': 0.3527302085219856,
  'mdape': np.float64(0.15432224238215392)},
 'test': {'r2': 0.4413369687070814,
  'mae': 287478.1881811353,
  'mape': 0.2526424417933166,
  'mdape': np.float64(0.15426469775353882)}}